# Job 4: Goodvibes referece

## 1. Select strong bragg peaks

In [1]:
% get the 400 most-intense Bragg reflections in this resolution bin
numRef = 400;

% choose reflections between 2.5 and 2.0A resolution
resMin = 2;
resMax = 2.5;

In [2]:
%%file selectStrongBraggPeaks.m

function hklRef = selectStrongBraggPeaks(hklTable,Basis,numRef,resMin,resMax)

% filter by reflection range
[sx,sy,sz] = Basis.invert.frac2lab(hklTable.h,hklTable.k,hklTable.l);
s = sqrt(sx.^2 + sy.^2 + sz.^2);
isIncl = 1./s >= resMin & 1./s <= resMax & ~isnan(hklTable.Fobs);
hklTable = hklTable(isIncl,:);

% sort by Fobs and choose the most intense entries
[~,ixorder] = sort(hklTable.Fobs,'descend');
hklTable = hklTable(ixorder(1:numRef),:);

hklRef = table2array(hklTable(:,{'h','k','l'}));

end

Created file '/nfs/chess/user/spm82/mdx/mac1_20251105_goodvibes/selectStrongBraggPeaks.m'.


In [3]:
%%file gridarray2hkl.m

function [h,k,l] = gridarray2hkl(hklGrid)

[h,k,l] = arrayfun(@(g) g.grid(),hklGrid,'Uni',0);
h = cell2mat(cellfun(@(v) shiftdim(v,-1),h,'uni',0));
k = cell2mat(cellfun(@(v) shiftdim(v,-1),k,'uni',0));
l = cell2mat(cellfun(@(v) shiftdim(v,-1),l,'uni',0));

end

Created file '/nfs/chess/user/spm82/mdx/mac1_20251105_goodvibes/gridarray2hkl.m'.


In [4]:
load proc/atomic_model.mat hklTable Basis

hklRef = selectStrongBraggPeaks(hklTable,Basis,numRef,resMin,resMax);

## 2. Load diffuse data around those peaks

In [5]:
mapFileName = 'export/mac1_5x5x11_map.h5';
dset = '/maps/total';

[hklGrid,I,sigma] = proc.script.LatticeDynamicsTools.loadBrillouinZones(mapFileName, dset, hklRef);


In [6]:
% Punch out the central voxel (set sigma to inf)
[n1,n2,n3] = hklGrid(1).invert.invert.frac2ind(0,0,0);
for n=1:size(sigma,1)
    sigma(n,n1,n2,n3) = inf;
end

In [9]:
% save the reference data

save('proc/reference_halos.mat','hklGrid','I','sigma');

## 3. One-phonon structure factor calculations for reference data points

Note: This step does not run on CLASSE jupyter hub without OOM errors. Let's write it to a file and execute in separate matlab process on compute farm node with sufficient memory.

In [10]:
%%file job04_goodvibes_reference_calc.m

[MT,rho] = proc.script.MapTools.import('export/mac1_edens.h5','unwrapped_cropped','rho');
load proc/goodvibes_model.mat ENM ResidueGroups M
load proc/reference_halos.mat hklGrid

% interpolate structure factors around reference peaks
[h,k,l] = gridarray2hkl(hklGrid);
supercell = hklGrid(1).invert.P;

LDT = proc.script.LatticeDynamicsTools('supercell',supercell,'Cell',ENM.Cell,'M',M);

sffun = LDT.calc1PSFInterpFromMap(rho,MT.Grid,MT.Basis);
[Gk,ind] = LDT.precompute1PSFs(sffun,h,k,l);

save proc/reference_calc.mat Gk ind LDT

Created file '/nfs/chess/user/spm82/mdx/mac1_20251105_goodvibes/job04_goodvibes_reference_calc.m'.
